# SpendShield Evidence-Driven ML Dataset Audit

This notebook audits the current SpendShield transaction data boundary before any ML model is implemented.

## Objective

- Verify the real MongoDB source, schema, volume, time coverage, and provenance.
- Determine whether a trustworthy target label exists.
- Define candidate features and explicitly exclude post-outcome/leaky fields.
- Establish a baseline-readiness gate without fabricating data, labels, or metrics.

## Non-goals

This notebook does not train a model, generate fraud labels, write to MongoDB, query Cassandra for current-state features, or make fraud/risk decisions. An anomaly signal must remain distinct from a fact or human-reviewed decision.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import random
import sys

SEED = 7
random.seed(SEED)

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "backend" / "app").is_dir():
            return candidate
    raise RuntimeError("SpendShield repository root was not found.")

REPO_ROOT = find_repo_root(Path.cwd())
BACKEND_ROOT = REPO_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

try:
    from dotenv import dotenv_values
    local_env = dotenv_values(BACKEND_ROOT / ".env")
except ImportError:
    local_env = {}

def configured(name: str, default: str | None = None) -> str | None:
    return os.getenv(name) or local_env.get(name) or default

MONGODB_URI = configured("SPENDSHIELD_MONGODB_URI")
MONGODB_DATABASE = configured("SPENDSHIELD_MONGODB_DATABASE", "spendshield")
AUDIT_RUN_AT = datetime.now(timezone.utc).isoformat()

{"seed": SEED, "repository_root": str(REPO_ROOT), "database": MONGODB_DATABASE, "uri_configured": bool(MONGODB_URI)}

{'seed': 7,
 'repository_root': 'D:\\E_Drive\\Project',
 'database': 'spendshield',
 'uri_configured': True}

## Data source and provenance policy

The authoritative source for this audit is the existing MongoDB transactions collection. The unit of analysis is one current transaction document whose status is COMPLETED. Cassandra is historical event storage and is not used to reconstruct current balances or create a competing training table.

The audit distinguishes SYNTHETIC, REAL_PUBLIC, SIMULATED_CASE, and USER_PROVIDED_AUTHORIZED_EVIDENCE. The current MongoDB operational data is labelled APPLICATION_OPERATIONAL_DATA; that label does not imply real-world financial data and does not provide a fraud ground truth.

In [2]:
from pymongo import MongoClient

mongo_client = None
transactions = None
connection = {
    "status": "BLOCKED",
    "source": "MongoDB current operational state",
    "database": MONGODB_DATABASE,
}

if MONGODB_URI:
    try:
        mongo_client = MongoClient(
            MONGODB_URI,
            serverSelectionTimeoutMS=3000,
            connectTimeoutMS=3000,
        )
        hello = mongo_client.admin.command("hello")
        database = mongo_client[MONGODB_DATABASE]
        transactions = database["transactions"]
        connection.update({
            "status": "AVAILABLE",
            "server_version": mongo_client.server_info().get("version"),
            "replica_set": hello.get("setName"),
            "is_writable_primary": bool(hello.get("isWritablePrimary")),
        })
    except Exception as exc:
        connection.update({"status": "BLOCKED", "error_type": type(exc).__name__})

connection

{'status': 'AVAILABLE',
 'source': 'MongoDB current operational state',
 'database': 'spendshield',
 'server_version': '8.3.4',
 'replica_set': 'rs0',
 'is_writable_primary': True}

In [3]:
COMPLETED_FILTER = {"status": "COMPLETED"}
REQUIRED_FIELDS = [
    "transaction_id", "user_id", "account_id", "timestamp",
    "amount", "currency", "status", "merchant_id",
    "category_id", "subcategory_id",
]
OPTIONAL_SNAPSHOT_FIELDS = ["merchant_name", "category_name", "subcategory_name"]
TARGET_CANDIDATE_FIELDS = [
    "fraud", "is_fraud", "label", "target", "human_decision", "review_outcome",
]

def field_exists_count(field: str) -> int:
    return transactions.count_documents({**COMPLETED_FILTER, field: {"$exists": True}})

def field_missing_count(field: str) -> int:
    return transactions.count_documents({**COMPLETED_FILTER, field: {"$exists": False}})

audit = {
    "audit_run_at": AUDIT_RUN_AT,
    "connection": connection,
    "provenance": "APPLICATION_OPERATIONAL_DATA",
    "collection": "transactions",
    "total_transaction_documents": None,
    "status_counts": {},
    "completed_transaction_count": 0,
    "missing_required_fields": {},
    "optional_snapshot_presence": {},
    "target_field_presence": {},
    "earliest_completed_timestamp": None,
    "latest_completed_timestamp": None,
}

if transactions is not None:
    audit["total_transaction_documents"] = transactions.count_documents({})
    audit["status_counts"] = {
        str(row["_id"]): int(row["count"])
        for row in transactions.aggregate([
            {"$group": {"_id": "$status", "count": {"$sum": 1}}}
        ])
    }
    audit["completed_transaction_count"] = transactions.count_documents(COMPLETED_FILTER)
    audit["missing_required_fields"] = {
        field: field_missing_count(field) for field in REQUIRED_FIELDS
    }
    audit["optional_snapshot_presence"] = {
        field: field_exists_count(field) for field in OPTIONAL_SNAPSHOT_FIELDS
    }
    audit["target_field_presence"] = {
        field: field_exists_count(field) for field in TARGET_CANDIDATE_FIELDS
    }
    earliest = transactions.find_one(COMPLETED_FILTER, {"timestamp": 1}, sort=[("timestamp", 1)])
    latest = transactions.find_one(COMPLETED_FILTER, {"timestamp": 1}, sort=[("timestamp", -1)])
    audit["earliest_completed_timestamp"] = str(earliest["timestamp"]) if earliest else None
    audit["latest_completed_timestamp"] = str(latest["timestamp"]) if latest else None

audit

{'audit_run_at': '2026-09-12T05:37:40.589853+00:00',
 'connection': {'status': 'AVAILABLE',
  'source': 'MongoDB current operational state',
  'database': 'spendshield',
  'server_version': '8.3.4',
  'replica_set': 'rs0',
  'is_writable_primary': True},
 'provenance': 'APPLICATION_OPERATIONAL_DATA',
 'collection': 'transactions',
 'total_transaction_documents': 0,
 'status_counts': {},
 'completed_transaction_count': 0,
 'missing_required_fields': {'transaction_id': 0,
  'user_id': 0,
  'account_id': 0,
  'timestamp': 0,
  'amount': 0,
  'currency': 0,
  'status': 0,
  'merchant_id': 0,
  'category_id': 0,
  'subcategory_id': 0},
 'optional_snapshot_presence': {'merchant_name': 0,
  'category_name': 0,
  'subcategory_name': 0},
 'target_field_presence': {'fraud': 0,
  'is_fraud': 0,
  'label': 0,
  'target': 0,
  'human_decision': 0,
  'review_outcome': 0},
 'earliest_completed_timestamp': None,
 'latest_completed_timestamp': None}

## Dataset contract and leakage boundary

The first candidate task is intentionally not selected as production ML. The contract records what could be evaluated after sufficient data exists.

Candidate observable features include amount, currency, payment-time merchant/category snapshots, and timestamps. Identifiers may be used for grouping or traceability but must not be treated as predictive evidence by default.

Fields such as status_history, balance_after, updated_at, failure_reason, event/publication identifiers, and any post-review decision are excluded from a point-in-time feature set because they may contain outcome or future information. A payment status is an operational outcome, not a fraud label.

In [4]:
DATASET_CONTRACT = {
    "dataset_id": "spendshield_completed_transactions_v1",
    "source": "MongoDB.transactions",
    "provenance": "APPLICATION_OPERATIONAL_DATA",
    "unit_of_analysis": "one completed current transaction document",
    "include_filter": {"status": "COMPLETED"},
    "candidate_features": [
        "amount", "currency", "timestamp", "merchant_id",
        "merchant_name", "category_id", "subcategory_id",
        "category_name", "subcategory_name",
    ],
    "traceability_fields": ["transaction_id", "account_id", "user_id"],
    "excluded_from_point_in_time_features": [
        "status_history", "balance_after", "updated_at",
        "failure_reason", "failure_code", "event_id",
        "publication_status", "human_decision", "review_outcome",
    ],
    "target": None,
    "target_status": "NOT_AVAILABLE_IN_CURRENT_OPERATIONAL_DATA",
    "currency_policy": "single-account-currency only; no implicit conversion",
    "cassandra_policy": "historical events are not a competing current-state dataset",
}

DATASET_CONTRACT

{'dataset_id': 'spendshield_completed_transactions_v1',
 'source': 'MongoDB.transactions',
 'provenance': 'APPLICATION_OPERATIONAL_DATA',
 'unit_of_analysis': 'one completed current transaction document',
 'include_filter': {'status': 'COMPLETED'},
 'candidate_features': ['amount',
  'currency',
  'timestamp',
  'merchant_id',
  'merchant_name',
  'category_id',
  'subcategory_id',
  'category_name',
  'subcategory_name'],
 'traceability_fields': ['transaction_id', 'account_id', 'user_id'],
 'excluded_from_point_in_time_features': ['status_history',
  'balance_after',
  'updated_at',
  'failure_reason',
  'failure_code',
  'event_id',
  'publication_status',
  'human_decision',
  'review_outcome'],
 'target': None,
 'target_status': 'NOT_AVAILABLE_IN_CURRENT_OPERATIONAL_DATA',
 'currency_policy': 'single-account-currency only; no implicit conversion',
 'cassandra_policy': 'historical events are not a competing current-state dataset'}

In [5]:
def baseline_boundary(audit_result: dict) -> dict:
    if audit_result["connection"]["status"] != "AVAILABLE":
        return {
            "status": "BLOCKED_DEPENDENCY",
            "reason": "MongoDB could not be read during the audit.",
            "metrics": None,
        }
    if audit_result["completed_transaction_count"] == 0:
        return {
            "status": "BLOCKED_INSUFFICIENT_DATA",
            "reason": "No completed transaction records are available.",
            "metrics": None,
        }
    if not any(audit_result["target_field_presence"].values()):
        return {
            "status": "NO_VERIFIED_TARGET",
            "reason": "No fraud or human-review target is present; supervised metrics are not valid.",
            "allowed_next_boundary": "descriptive or unsupervised baseline evaluation only after data-quality review",
            "metrics": None,
        }
    return {
        "status": "DATA_REVIEW_REQUIRED",
        "reason": "A candidate target exists but requires provenance and label-quality review before training.",
        "metrics": None,
    }

baseline = baseline_boundary(audit)
baseline

{'status': 'BLOCKED_INSUFFICIENT_DATA',
 'reason': 'No completed transaction records are available.',
 'metrics': None}

In [6]:
report = {
    "notebook": "01_dataset_audit.ipynb",
    "audit_status": (
        "INSUFFICIENT_DATA"
        if audit["completed_transaction_count"] == 0 and connection["status"] == "AVAILABLE"
        else connection["status"]
    ),
    "audit": audit,
    "dataset_contract": DATASET_CONTRACT,
    "baseline_boundary": baseline,
    "model_training_performed": False,
    "metrics_claimed": False,
    "fraud_decision_created": False,
}

print(json.dumps(report, indent=2, default=str))

if mongo_client is not None:
    mongo_client.close()

{
  "notebook": "01_dataset_audit.ipynb",
  "audit_status": "INSUFFICIENT_DATA",
  "audit": {
    "audit_run_at": "2026-09-12T05:37:40.589853+00:00",
    "connection": {
      "status": "AVAILABLE",
      "source": "MongoDB current operational state",
      "database": "spendshield",
      "server_version": "8.3.4",
      "replica_set": "rs0",
      "is_writable_primary": true
    },
    "provenance": "APPLICATION_OPERATIONAL_DATA",
    "collection": "transactions",
    "total_transaction_documents": 0,
    "status_counts": {},
    "completed_transaction_count": 0,
    "missing_required_fields": {
      "transaction_id": 0,
      "user_id": 0,
      "account_id": 0,
      "timestamp": 0,
      "amount": 0,
      "currency": 0,
      "status": 0,
      "merchant_id": 0,
      "category_id": 0,
      "subcategory_id": 0
    },
    "optional_snapshot_presence": {
      "merchant_name": 0,
      "category_name": 0,
      "subcategory_name": 0
    },
    "target_field_presence": {
      "fr

## Decision

The notebook is complete when its audit output is reproducible and its readiness decision is evidence-based. A BLOCKED_INSUFFICIENT_DATA or NO_VERIFIED_TARGET result is a valid result; it is not a failed model run.

The next ML step is to obtain an explicitly labelled, provenance-documented dataset or a deliberately marked synthetic/demo dataset. That dataset must be kept separate from MongoDB current operational state, split temporally where appropriate, and evaluated against a simple baseline before any model artifact or inference route is considered.